In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv

if os.environ['AICREDITS_API_KEY']:
    print("AICredits API Key is set.")
else:
    raise ValueError("AICredits API Key is not set.")

AICredits API Key is set.


In [2]:
llm = ChatOpenAI(
    model="openai/gpt-5-nano",
    base_url="https://api.aicredits.in/v1",
    api_key=os.environ["AICREDITS_API_KEY"],
)
llm.invoke("I want to know the meaning of water").content

'Do you mean the literal substance called water, or the word’s meaning in a broader or symbolic sense? Here are both:\n\n1) Water as a substance\n- What it is: Water is a chemical compound with the formula H2O (two hydrogen atoms and one oxygen atom).\n- Important properties: It is a liquid at most Earth-ambient temperatures, a good solvent, and it can exist as a solid (ice), liquid, or gas (steam). It’s polar and forms hydrogen bonds, which give it unique properties like high surface tension and the ability to dissolve many substances.\n- What it does: It’s essential for most forms of life; it covers about 71% of Earth’s surface; used for drinking, farming, cleaning, industrial processes, cooling, etc.\n- Simple facts: Freezing point 0°C (at 1 atm); boiling point 100°C (at 1 atm); density about 1 g/cm³ when liquid.\n\n2) Water in a symbolic or figurative sense\n- Common symbolic meanings: life and renewal, cleansing or purification, change and flow, emotions (tears), adaptability, and

### Pydantic Schema

In [3]:
from pydantic import BaseModel, Field

class graph_schema(BaseModel):
    topic: str = Field(description="The topic of the graph")
    post: str = Field(description="The LinkedIn post content")
    curated_post: str = Field(description="The curated LinkedIn post content")

demo_obj = graph_schema(
    topic="The importance of Learning how to learn",
    post="Learning how to learn is crucial skill for life. It helps everyone to face unimaginable problems in life and come up with unique solutions",
    curated_post="In this age of cut throat competition. Learning how to learn is more important than ever."
)
print(demo_obj)

topic='The importance of Learning how to learn' post='Learning how to learn is crucial skill for life. It helps everyone to face unimaginable problems in life and come up with unique solutions' curated_post='In this age of cut throat competition. Learning how to learn is more important than ever.'


In [4]:
def create_post(state: graph_schema) -> graph_schema:
    state = state.model_dump()
    topic = state['topic']
    post = llm.invoke(f"Write a linkedIn post about {topic}").content
    state['post'] = post
    return state

def curate_post(state: graph_schema) -> graph_schema:
    state = state.model_dump()
    post = state['post']
    curated_post = llm.invoke(f"Curate the following LinkedIn post with GenZ tone: {post}").content
    state['curated_post'] = curated_post
    return state

In [5]:
from langgraph.graph import StateGraph, START, END
graph = StateGraph(graph_schema)
graph.add_node("create_post", create_post)
graph.add_node("curate_post", curate_post)

graph.add_edge(START, "create_post")
graph.add_edge("create_post", "curate_post")
graph.add_edge("curate_post", END)

pydantic_graph = graph.compile()
print(pydantic_graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	create_post(create_post)
	curate_post(curate_post)
	__end__([<p>__end__</p>]):::last
	__start__ --> create_post;
	create_post --> curate_post;
	curate_post --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [6]:
pydantic_graph.invoke(
    {"topic":"The importance of learning how to learn",
     "post":"",
     "curated_post":""}
)

{'topic': 'The importance of learning how to learn',
 'post': 'The most valuable skill in a world that changes faster than ever isn’t what you know—it’s how you learn. Learning how to learn (meta-learning) is what lets you pick up new tools, adapt to new roles, and stay ahead long after formal education ends.\n\nKey ideas to practice:\n- Start with a clear goal: define what you want to learn and why it matters.\n- Deconstruct the skill: break it into sub-skills and prerequisites; map a learning path.\n- Choose active methods: leverage retrieval practice, deliberate practice, teaching others, and real-world application—not just passive reading.\n- Build a tiny, repeatable plan: time-boxed sessions, micro-habits, quick wins.\n- Seek quick feedback and iterate: use quizzes, feedback loops, and small projects to apply what you’ve learned.\n- Reflect and adapt: regularly ask what worked, what didn’t, and how to adjust your approach.\n- Apply immediately: find a real project or problem to co